In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 92.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
import os
import re
import pdfplumber
import pandas as pd
import numpy as np
from datetime import datetime

###Extract Text from PDF


In [ ]:
def extract_information(uploaded_file):

    text = ""

    with pdfplumber.open(uploaded_file) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

###Extract Structured Information Using Regex

In [ ]:
def extract_details(resume_text):

    text = str(resume_text)
    text = text.replace("’", "'")

    # ---------------- SKILLS ----------------

    skills = []

    tech_match = re.search(
        r'technical\s*skills\s*:?(.*?)(?=soft\s*skills|languages|education|experience|projects|certifications|$)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    soft_match = re.search(
        r'soft\s*skills\s*:?(.*?)(?=languages|education|experience|projects|certifications|$)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    skills_match = re.search(
        r'(?:^|\n)\s*(?:skills|key\s*skills|core\s*skills)\s*:?(.*?)(?=languages|education|experience|projects|certifications|$)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    all_skills_text = ""

    if tech_match:
        all_skills_text += "\n" + tech_match.group(1)

    if soft_match:
        all_skills_text += "\n" + soft_match.group(1)

    if skills_match:
        all_skills_text += "\n" + skills_match.group(1)

    if all_skills_text:

        raw_skills = [
            s.strip()
            for s in re.split(r'[,;\n]+', all_skills_text)
            if s.strip()
        ]

        remove_patterns = [
            r'^technical\s*skills:?$',
            r'^soft\s*skills:?$',
            r'^skills:?$',
            r'^o$',
            r'^●$',
            r'^▪$',
            r'^_$'
        ]

        clean_skills = []

        for skill in raw_skills:
          # remove bullets
          skill = re.sub(r'^[o●▪•]+\s*', '', skill)

          # remove separator lines
          if set(skill.replace('_', '')) == {''}:
              continue

          # remove category labels
          skip = False
          for pattern in remove_patterns:
              if re.match(pattern, skill, re.IGNORECASE):
                  skip = True
                  break

          if skip:
              continue

          skill = skill.strip(" .:-")

          if skill:
              clean_skills.append(skill)

        skills = list(dict.fromkeys(clean_skills))

    # ---------------- EDUCATION ----------------
    degree = "Not Specified"
    specialization = "Not Specified"

    edu_match = re.search(
        r'education\s*:?\s*(.*?)(?=experience|projects|skills|certifications|languages|$)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    education = edu_match.group(1).strip() if edu_match else ""

    degree_patterns = [
        r"(Bachelor(?:'s)?|Master(?:'s)?)\s+Degree\s+in\s+([A-Za-z\s&-]+)",
        r"(Bachelor(?:'s)?|Master(?:'s)?)\s+of\s+[A-Za-z\s&-]+\s+in\s+([A-Za-z\s&-]+)",
        r"(B\.?Sc\.?|M\.?Sc\.?|BS|MS)(?:\s+in\s+)?([A-Za-z\s&-]+)"

    ]


    for pattern in degree_patterns:
      match = re.search(pattern, education, re.IGNORECASE)

      if not match and education:
        match = re.search(pattern, text, re.IGNORECASE)

      if match:

        degree = match.group(1).strip()
        specialization = match.group(2).strip()

        # Normalize abbreviations
        degree_map = {
            "BSc": "Bachelor's",
            "BS": "Bachelor's",
            "MSc": "Master's",
            "MS": "Master's"
        }

        degree = degree_map.get(
            degree.replace(".", ""),
            degree
        )

        break

    # Clean specialization noise
    if specialization:
        specialization = re.split(
            r'\n|University|Institute|College|Workshop|Courses',
            specialization,
            flags=re.IGNORECASE
        )[0].strip()

    # ---------------- PROJECTS ----------------
    projects = "No Projects"

    project_match = re.search(
        r'(?:^|\n)\s*projects\s*:?\s*(.*?)(?=\n\s*(experience|education|skills|certifications|languages|achievements)\s*:?\s*|$)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    if project_match:
        projects = project_match.group(1).strip()

    # ---------------- EXPERIENCE ----------------

    experience_years = 0

    date_pattern = r'([A-Za-z]{3})\s+(\d{4})\s*[–-]\s*(?:([A-Za-z]{3})\s+(\d{4})|(Present|Current))'

    matches = re.findall(
        date_pattern,
        text,
        re.IGNORECASE
    )

    month_map = {
        "jan": 1, "feb": 2, "mar": 3,
        "apr": 4, "may": 5, "jun": 6,
        "jul": 7, "aug": 8, "sep": 9,
        "oct": 10, "nov": 11, "dec": 12
    }

    intervals = []

    for start_month, start_year, end_month, end_year, current_flag in matches:

        start = datetime(
            int(start_year),
            month_map[start_month.lower()],
            1
        )

        if current_flag:
            end = datetime.today()
        else:
            end = datetime(
                int(end_year),
                month_map[end_month.lower()],
                1
            )

        intervals.append((start, end))

    if intervals:

        intervals.sort(key=lambda x: x[0])

        merged = [intervals[0]]

        for current in intervals[1:]:

            last_start, last_end = merged[-1]

            if current[0] <= last_end:
                merged[-1] = (
                    last_start,
                    max(last_end, current[1])
                )
            else:
                merged.append(current)

        total_months = 0

        for start, end in merged:

            months = (
                (end.year - start.year) * 12
                + (end.month - start.month)
            )

            total_months += months

        experience_years = round(total_months / 12, 1)

    return {
        "Education": degree,
        "Specialization": specialization,
        "Experience_Years": experience_years,
        "Projects": projects,
        "Skills": skills
    }